# BRAM AXI Test for PYNQ-Z1

This notebook tests the dual-port BRAM design with:
- AXI BRAM Controller for Python access
- FSM that reads bottom 4 locations and stores sum at top
- GPIO control for FSM start/status

In [ ]:
from pynq import Overlay, MMIO
import time

# Load the overlay
overlay = Overlay('/home/xilinx/overlays/lab4.bit')

# BRAM is Memory type - use directly (has built-in read/write via mmio)
bram_ctrl = overlay.axi_bram_ctrl_0

# GPIO is AxiGPIO type - use directly
gpio = overlay.axi_gpio_0

BRAM_SIZE = 1024  # 1024 words = 4KB
TOP_ADDR = 1023

print(overlay)
print(f"BRAM: {bram_ctrl}")
print(f"GPIO: {gpio}")
print("Overlay loaded!")

In [ ]:
def is_fsm_done():
    """Read done bit from GPIO Channel 2 (input, 1-bit)"""
    return gpio.channel2.read() & 0x1 != 0

def fsm_reset(assert_reset):
    """Control FSM reset via GPIO Channel 1 (output).
    assert_reset=True  → hold FSM in reset (IDLE)
    assert_reset=False → release reset, FSM auto-runs to DONE
    """
    gpio.channel1.write(1 if assert_reset else 0, 0x1)

def run_fsm(timeout_ms=1000):
    """Assert then deassert reset to trigger one FSM run. Wait for done."""
    fsm_reset(True)
    time.sleep(0.001)
    fsm_reset(False)
    start_time = time.time()
    while True:
        if is_fsm_done():
            return True
        if time.time() - start_time > timeout_ms / 1000.0:
            return False
        time.sleep(0.001)

def write_bram_word(address, data):
    """Write 32-bit word to BRAM address (word-indexed)"""
    if address < 0 or address >= BRAM_SIZE:
        raise ValueError(f"Address {address} out of range (0-{BRAM_SIZE-1})")
    bram_ctrl.write(address * 4, data)

def read_bram_word(address):
    """Read 32-bit word from BRAM address (word-indexed)"""
    if address < 0 or address >= BRAM_SIZE:
        raise ValueError(f"Address {address} out of range (0-{BRAM_SIZE-1})")
    return bram_ctrl.read(address * 4)

## Test 1: Direct BRAM Access

In [ ]:
# Test direct BRAM access
test_addr = 100
test_data = 0xDEADBEEF

print(f"Writing 0x{test_data:08X} to address {test_addr}")
write_bram_word(test_addr, test_data)

read_data = read_bram_word(test_addr)
print(f"Read from address {test_addr}: 0x{read_data:08X}")

if read_data == test_data:
    print("✓ Direct BRAM access test PASSED!")
else:
    print("✗ Direct BRAM access test FAILED!")

## Test 2: FSM Operation

### Initialize BRAM with Test Data

In [ ]:
# Test values for addresses 0-3
#test_values = [0x12345678, 0xABCDEF00, 0x11223344, 0x55667788]
test_values = [0x01000004, 0x00100003, 0x10000002, 0x00000001]


print("=== Writing Initial Values ===")
for i, value in enumerate(test_values):
    print(f"Writing to address {i}: 0x{value:08X}")
    write_bram_word(i, value)

print("\n=== Verifying Initial Write ===")
write_success = True
for i, expected in enumerate(test_values):
    actual = read_bram_word(i)
    status = "✓" if actual == expected else "✗"
    print(f"Reading address {i}: 0x{actual:08X} {status}")
    if actual != expected:
        write_success = False

if write_success:
    print("\n✓ Initial write to addresses 0-3 successful!")
else:
    print("\n✗ Initial write verification FAILED!")

### Run FSM Test

In [ ]:
# Hold FSM in reset before run
fsm_reset(True)
time.sleep(0.01)

done = is_fsm_done()
print(f"FSM in reset - Done: {done}")
print("✓ FSM held in reset, ready to run")

In [ ]:
print("=== Running FSM ===")
print("Asserting reset then releasing to trigger run...")

success = run_fsm()

if success:
    print("✓ FSM completed (done=1)")
else:
    print("✗ FSM timed out!")

print(f"Done: {is_fsm_done()}")

### Verify Results

In [ ]:
print("=== BRAM Contents After FSM ===")

for i in range(4):
    value = read_bram_word(i)
    print(f"Address {i}: 0x{value:08X}")

result = read_bram_word(TOP_ADDR)
print(f"Address {TOP_ADDR}: 0x{result:08X}")

expected_sum = sum(test_values) & 0xFFFFFFFF
print(f"\nExpected sum: 0x{expected_sum:08X}")
print(f"Actual result: 0x{result:08X}")

if result == expected_sum:
    print("✓ Sum calculation correct!")
else:
    print("✗ Sum calculation incorrect!")

# Check done is still high (FSM stuck in DONE until reset)
if is_fsm_done():
    print("✓ FSM stuck in DONE state (as expected)")
else:
    print("✗ Done unexpectedly cleared")

## Test Summary

In [ ]:
print("\n=== Test Summary ===")
print("1. Direct BRAM access: " + ("Working" if write_success else "Failed"))
print("2. FSM operation: " + ("Working" if success and result == expected_sum else "Failed"))

if write_success and success and result == expected_sum:
    print("\n✓ All tests PASSED!")
else:
    print("\n✗ Some tests FAILED!")